## Librerias

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Carga de datos

In [7]:
df = pd.read_csv("../../Data/clean_data_16-03-2026.csv", parse_dates=['insert_date','first_review_date','last_review_date'])
df

,apartment_id,name,description,host_id,neighbourhood_name,neighbourhood_district,room_type,accommodates,bathrooms,bedrooms,...,review_scores_communication,review_scores_location,review_scores_value,is_instant_bookable,reviews_per_month,country,city,insert_date,price_missing,reviews 80+
0,11964,A ROOM WITH A VIEW,Private bedroom in our attic apartment. Right ...,45553,Centro,(sin contestar),Private room,2,2.0,1.0,...,100.0,100.0,100.0,False,75.0,spain,Malaga,2018-07-31,False,True
1,21853,Bright and airy room,We have a quiet and sunny room with a good vie...,83531,C�rmenes,Latina,Private room,1,1.0,1.0,...,100.0,80.0,90.0,False,52.0,spain,Madrid,2020-01-10,False,True
2,32347,Explore Cultural Sights from a Family-Friendly...,Open French doors and step onto a plant-filled...,139939,San Vicente,Casco Antiguo,Entire home/apt,4,1.0,2.0,...,100.0,100.0,100.0,True,142.0,spain,Sevilla,2019-07-29,False,True
3,35379,Double 02 CasanovaRooms Barcelona,Room at a my apartment. Kitchen and 2 bathroom...,152232,l'Antiga Esquerra de l'Eixample,Eixample,Private room,2,2.0,1.0,...,100.0,100.0,90.0,True,306.0,spain,Barcelona,2020-01-10,False,True
4,35801,Can Torras Farmhouse Studio Suite,Lay in bed & watch sunlight change the mood of...,153805,Quart,(sin contestar),Private room,5,1.0,2.0,...,100.0,100.0,100.0,False,39.0,spain,Girona,2019-02-19,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7688,32392193,Espectacular habitaci�n,(sin contestar),238089984,Sant Antoni,Eixample,Private room,1,1.0,1.0,...,NaN,NaN,NaN,True,NaN,spain,Barcelona,2019-10-16,False,NaN
7689,32392774,? Tu Hogar de Lujo ????? en el Centro de Sevilla,"Exclusivo, amplio y luminoso alojamiento situa...",243246681,Arenal,Casco Antiguo,Entire home/apt,6,2.0,3.0,...,100.0,100.0,100.0,False,157.0,spain,Sevilla,2021-01-31,False,True
7690,32395123,Rooms by G Bella Mar�a 3,The 2-star Bella Maria has 24-hourreception an...,159933359,Felanitx,(sin contestar),Entire home/apt,2,1.0,1.0,...,NaN,NaN,NaN,True,NaN,spain,Mallorca,2019-04-24,False,NaN
7691,32407332,LUMINOSO Y ENCANTADOR PISO CERCA DE TODO,PISO MUY ILUMINADO CON UNA TERRAZA ESTUPENDA C...,187631805,Proven�als del Poblenou,Sant Mart�,Private room,3,2.0,2.0,...,100.0,100.0,100.0,True,389.0,spain,Barcelona,2019-08-12,False,True


## Operaciones
- Tasa de ocupación mensual
- Ciudad con mayor ocupación (mensual)

#### Preparación del dataset

In [15]:
df_operaciones = df.copy()

availability_cols = [
    "availability_30",
    "availability_60",
    "availability_90",
    "availability_365"
]

# apartamentos totalmente ocupados
df_operaciones["fully_booked"] = (df_operaciones[availability_cols] == 0).all(axis=1)


#### Variables derivadas (ocupación)

In [16]:
df_operaciones["occ_30"] = (30 - df_operaciones["availability_30"]) / 30
df_operaciones["occ_60"] = (60 - df_operaciones["availability_60"]) / 60
df_operaciones["occ_90"] = (90 - df_operaciones["availability_90"]) / 90
df_operaciones["occ_365"] = (365 - df_operaciones["availability_365"]) / 365

occupancy_cols = ["occ_30", "occ_60", "occ_90", "occ_365"]

# días ocupados (para KPI principal)
df_operaciones["occupied_days_30"] = 30 - df_operaciones["availability_30"]

#### KPI globales operaciones

In [17]:
# inventario total
total_apartments = df_operaciones["apartment_id"].nunique()

# alojamientos con disponibilidad en próximos 30 días
available_30 = (df_operaciones["availability_30"] > 0).sum()

# ratio de oferta disponible
available_supply_ratio = round((available_30 / active_apartments) * 100, 2)

# apartamentos completamente reservados
fully_booked_active = df_operaciones[
    (df_operaciones["fully_booked"]) &
    (df_operaciones["has_availability"])
].shape[0]

# ocupación promedio por horizonte temporal
occupancy_rates = (df_operaciones[occupancy_cols].mean() * 100).round(2)

#### KPI por segmento

In [18]:
# ocupación por ciudad
occupancy_by_city = (
    df_operaciones
    .groupby("city")[occupancy_cols]
    .mean()
    .mul(100)
    .round(2)
    .reset_index()
)

# ciudad con mayor ocupación mensual
top_city = (
    occupancy_by_city
    .sort_values("occ_30", ascending=False)
    .iloc[0]
)

# oferta disponible por ciudad
supply_by_city = (
    df_operaciones[df_operaciones["availability_30"] > 0]
    .groupby("city")["apartment_id"]
    .nunique()
    .reset_index(name="available_listings")
    .sort_values(by="available_listings", ascending=False)
)

# oferta disponible por tipo de habitación
supply_by_room = (
    df_operaciones[df_operaciones["availability_30"] > 0]
    .groupby("room_type")["apartment_id"]
    .nunique()
    .reset_index(name="available_listings")
    .sort_values(by="available_listings", ascending=False)
)

# catálogo no disponible actual
not_available = total_apartments - active_apartments
not_available_pct = round((not_available / total_apartments) * 100, 2)

# semana pasada
prev_total = 6615
prev_active = 6087

prev_not_available = prev_total - prev_active
prev_not_available_pct = round((prev_not_available / prev_total) * 100, 2)

print("No disponible actual:", not_available, f"({not_available_pct}%)")
print("No disponible antes:", prev_not_available, f"({prev_not_available_pct}%)")


No disponible actual: 534 (6.94%)
No disponible antes: 528 (7.98%)


#### Tabla final de KPI operaciones

In [19]:
kpi_operaciones = {
    "total_apartments": int(total_apartments),
    "active_apartments": int(active_apartments),
    "available_next_30_days": int(available_30),
    "available_supply_%": float(available_supply_ratio),
    "fully_booked_active": int(fully_booked_active),
    "monthly_occupancy_%": float(monthly_occupancy_rate),
    "yearly_occupancy_%": float(occupancy_rates["occ_365"]),
    "top_city_by_occupancy": top_city["city"],
    "top_city_occupancy_%": float(top_city["occ_30"])
}

kpi_operaciones_df = pd.DataFrame(
    list(kpi_operaciones.items()),
    columns=["KPI", "Valor"]
)

kpi_operaciones_df

,KPI,Valor
0,total_apartments,7693
1,active_apartments,7159
2,available_next_30_days,5450
3,available_supply_%,76.13
4,fully_booked_active,919
5,monthly_occupancy_%,63.24
6,yearly_occupancy_%,48.59
7,top_city_by_occupancy,Madrid
8,top_city_occupancy_%,64.07
